In [ ]:
import pandas as pd
import os
import io
import re
from io import BytesIO
from ftplib import FTP
from dotenv import load_dotenv
load_dotenv()

In [42]:
def ConnectFTP(server, username, password):
    """
    Descripcion: Esta funcion establece una conexion FTP con el servidor especificado.
    Parametros:
    server (str): La direccion del servidor FTP.
    username (str): El nombre de usuario para la conexion FTP.
    password (str): La contrasena para la conexion FTP.
    Retorna: Un objeto FTP conectado al servidor.
    """
    ftp = FTP(timeout=60)                
    ftp.connect(server, 21)               
    ftp.login(user=username, passwd=password)

    ftp.set_pasv(True)                    
    ftp.voidcmd("TYPE I")                 

    print(f"Conectado a {server}")
    return ftp

def UploadCsvToFTP(df, path):
    """
    Descripcion: Esta funcion sube un DataFrame de pandas como un archivo CSV a un servidor FTP.
    Parametros:
    df (DataFrame): El DataFrame que se desea subir.
    path (str): La ruta en el servidor FTP donde se guardará el archivo CSV.
    Retorna: None
    """
    csv_buffer = io.BytesIO()
    df.to_csv(csv_buffer, index=False, encoding='utf-8')
    csv_buffer.seek(0)
    ftp = ConnectFTP(os.getenv('ftp_server'),os.getenv('ftp_user'),os.getenv('ftp_password'))
    # remote_path = f"/pruebas/csv/{path}"
    ftp.storbinary(f"STOR {path}", csv_buffer)
    ftp.quit()
    print("Archivo subido correctamente:", path)

def ReadCsvFromFTP(remote_file_path):
    '''
    Descripcion: Esta funcion lee un archivo CSV desde un servidor FTP y lo carga en un DataFrame de pandas.
    Parametros:
    ftp (FTP): Un objeto FTP conectado al servidor.
    remote_file_path (str): La ruta del archivo CSV en el servidor FTP.
    Retorna: Un DataFrame de pandas que contiene los datos del archivo CSV.
    '''
    ftp = ConnectFTP(os.getenv('ftp_server'), os.getenv('ftp_user'), os.getenv('ftp_password'))
    with BytesIO() as bio:
        ftp.retrbinary(f'RETR {remote_file_path}', bio.write)
        bio.seek(0)
        df = pd.read_csv(bio)
    return df

In [43]:
def gen_valueof_column():
    '''
    Description: Esta funcion genera un diccionario que mapea los nombres de las sustancias a sus respectivas claves.
    Returns:
    dict: Un diccionario que contiene los nombres de las sustancias como claves y sus respectivas claves como valores.
    '''
    return {
        'Tabaco': 'Tabaco',
        'Alcohol': 'Alcohol',
        'Marihuana': 'Marihuana',
        'Cocaína': 'Cocaína',
        'Metanfetaminas': 'Metanfetaminas',
        'Opioides': 'Opioides',
        'Inhalables': 'Inhalables',
        'Alucinógenos': 'Alucinógenos',
        'Medicamentos': 'Medicamentos',
        'Nuevas Sustancias Psicoactivas': 'Nuevas Sustancias Psicoactivas',
        'Otras Sustancias': 'Otras Sustancias',
    }

def gen_valueof_column_sust():
    '''
    Description: Esta funcion genera un diccionario que mapea los nombres de las sustancias a sus respectivas claves en español.
    Returns:
    dict: Un diccionario que contiene los nombres de las sustancias como claves y sus respectivas claves en español como valores.
    '''
    return {
    'Tabaco': 'Tabaco',
    'Nicotina en dispositivo electrónico': 'Nicotina en dispositivo electrónico',
    'Bebidas Fermentadas': 'Bebidas Fermentadas',
    'Bebidas Destiladas': 'Bebidas Destiladas',
    'Alcohol puro (alcohol del 96)': 'Alcohol puro (alcohol del 96)',
    'Mariguana estándar': 'Mariguana estándar',
    'Mariguana modificada, con alto nivel de THC': 'Mariguana modificada, con alto nivel de THC',
    'Hachís - Resina': 'Hachís - Resina',
    'Hachís - Aceite o miel': 'Hachís - Aceite o miel',
    'Mariguana sintética': 'Mariguana sintética',
    'Medicamentos con compuestos cannábicos': 'Medicamentos con compuestos cannábicos',
    'Cocaína en polvo blanco': 'Cocaína en polvo blanco',
    'Crack': 'Crack',
    'Basuco (pasta base de cocaína)': 'Basuco (pasta base de cocaína)',
    'Otras presentaciones de cocaína': 'Otras presentaciones de cocaína',
    'Metanfetaminas(cristal)': 'Metanfetaminas(cristal)',
    'Metanfetaminas(hielo)': 'Metanfetaminas(hielo)',
    'MetanfetaminasMetas': 'MetanfetaminasMetas',
    'Sales de baño': 'Sales de baño',
    'Anfetaminas y derivados': 'Anfetaminas y derivados',
    'Anorexígenos': 'Anorexígenos',
    'Drogas de diseño con efectos estimulantes': 'Drogas de diseño con efectos estimulantes',
    'Otros estimulantes': 'Otros estimulantes',
    'Solventes o removedores': 'Solventes o removedores',
    'Pegamentos': 'Pegamentos',
    'Esmaltes y pinturas': 'Esmaltes y pinturas',
    'Gasolinas y combustibles': 'Gasolinas y combustibles',
    'Otros inhalantes': 'Otros inhalantes',
    'LSD': 'LSD',
    'LSA': 'LSA',
    'Hongos alucinógenos': 'Hongos alucinógenos',
    'Peyote (mescalina)': 'Peyote (mescalina)',
    'Salvia divinorum u otras variedades de salvia': 'Salvia divinorum u otras variedades de salvia',
    'Ololiuhqui': 'Ololiuhqui',
    'Floripondio': 'Floripondio',
    'Ayahuasca': 'Ayahuasca',
    'Otras plantas con efectos alucinógenos o disociativos': 'Otras plantas con efectos alucinógenos o disociativos',
    'Fenciclidina (PCP, polvo de ángel)': 'Fenciclidina (PCP, polvo de ángel)',
    'Análogos del PCP': 'Análogos del PCP',
    'Ketamina': 'Ketamina',
    'DMT': 'DMT',
    'Triptaminas psicodélicas derivadas y análogas del DMT': 'Triptaminas psicodélicas derivadas y análogas del DMT',
    'Grupo 2Cx': 'Grupo 2Cx',
    'Grupo DOx': 'Grupo DOx',
    'Grupo NBOMe': 'Grupo NBOMe',
    'Otras drogas de diseño con efectos psicodélicos': 'Otras drogas de diseño con efectos psicodélicos',
    'Otras sustancias con efectos psicodélicos': 'Otras sustancias con efectos psicodélicos',
    'Éxtasis (MDMA)': 'Éxtasis (MDMA)',
    'Drogas de diseño con efectos similares al éxtasis': 'Drogas de diseño con efectos similares al éxtasis',
    'Piperazinas con efectos similares al éxtasis': 'Piperazinas con efectos similares al éxtasis',
    'Aminoindanos': 'Aminoindanos',
    'Benzodiacepinas y sustancias relacionadas': 'Benzodiacepinas y sustancias relacionadas',
    'Rohypnol': 'Rohypnol',
    'GHB o similares': 'GHB o similares',
    'Sedantes hipnóticos': 'Sedantes hipnóticos',
    'Otros depresores': 'Otros depresores',
    'Opio': 'Opio',
    'Morfina(uso fuera de prescripción)': 'Morfina(uso fuera de prescripción)',
    'Codeína(uso fuera de prescripción)': 'Codeína(uso fuera de prescripción)',
    'Heroína blanca': 'Heroína blanca',
    'Heroína negra': 'Heroína negra',
    'Oxicodona': 'Oxicodona',
    'Opiodes sintéticos (uso fuera de prescripción)': 'Opiodes sintéticos (uso fuera de prescripción)',
    'Fentanilo y análogos (uso fuera de prescripción)': 'Fentanilo y análogos (uso fuera de prescripción)',
    'Dextrometorfano (uso fuera de prescripción)': 'Dextrometorfano (uso fuera de prescripción)',
    'Derivados opiodes combinados con analgésicos': 'Derivados opiodes combinados con analgésicos',
    'Desomorfina': 'Desomorfina',
    'Anticolinérgicos': 'Anticolinérgicos',
    'Antiespasmódicos': 'Antiespasmódicos',
    'Antiparkinsonianos': 'Antiparkinsonianos',
    'Anticonvulsivos': 'Anticonvulsivos',
    'Antidepresivos': 'Antidepresivos',
    'Antipsicóticos': 'Antipsicóticos',
    'Antihistamínicos': 'Antihistamínicos',
    'Otras sustancias de abuso con utilidad médica': 'Otras sustancias de abuso con utilidad médica',
    'Nitritos': 'Nitritos',
    'Anestésicos': 'Anestésicos',
    'Bebidas energizantes': 'Bebidas energizantes',
    'Bebidas inteligentes': 'Bebidas inteligentes',
    'Esteroides anabólicos': 'Esteroides anabólicos',
    'Sustancias no clasificadas': 'Sustancias no clasificadas',
    'Sustancias sin especificar': 'Sustancias sin especificar'
}

def extract_dem(denominador):
    '''
    Description: Esta funcion extrae el nombre de la columna que contiene los valores de las sustancias a partir del denominador.
    Parameters:
    denominador (str): El nombre del denominador que se utilizara para extraer el nombre de la columna.
    Returns:
    dict: Un diccionario que contiene el nombre de la columna que contiene los valores de las sustancias.
    '''
    if 'Sust' in denominador:
        value_to_column = gen_valueof_column_sust()
    else:
        value_to_column = gen_valueof_column() 
    return value_to_column

def DropedColumns(df):
    '''
    Description: Esta funcion elimina columnas especificas del DataFrame que no son necesarias para el analisis.
    Parameters:
    df (DataFrame): Un DataFrame de pandas que contiene los datos.
    Returns:
    DataFrame: Un DataFrame de pandas con las columnas especificas eliminadas.
    '''
    elimin = ['ComunTipoUsoDrogaIlicitaId','FechaRegistro', 'EIP_FolioId', 'EntrevistaInicialPacienteId', 'ComunComunidadLGBTTTIId', 'ComunidadIndigena', 'HablaLenguaIndigena', 'DiscapacidadPerceptualDescripcion', 'LugarNacimientoId', 'ComunPrimeraVezId', 'ComunEmbarazoId', 'EmbarazoSemanas', 'VividoEstadosUnidos', 'ActividadLaboral', 'ComunDiscapacidadPerceptualId', 'PaisId', 'UltimosDoceMesesPaisId', 'EstadoEUAId', 'EIRL_FolioId', 'EntrevistaInicialResponsableLegalId', 'ComunTipoRelacionId', 'ComunTipoRelacionClaveId', 'EISP_FolioId', 'EntrevistaInicialSustanciaPatronId', 'PreferidaSustanciaId', 'MultipleUsoMismoDia', 'ComunFrecuenciaAlcoholId', 'ComunFrecuenciaDrogasId','EIM_FolioId',
                'SustanciaComb1', 'SustanciaComb2', 'SustanciaComb3', 'SustanciaComb4', 'SustanciaComb5', 'SustanciaComb6', 'SustanciaComb7', 'SustanciaComb8', 'SustanciaComb9', 'SustanciaComb10', 'TipoServicioId', 'TipoPacienteId', 'FuenteReferenciaId', 'ComunAsistioPacienteId','NumeroFamiliares', 'ComunAtencionResidencialId', 'CURP', 'ConvenioId', 'FechaAtencion', 'EntidadClave' ,'EntidadClave.1', 'EntidadClave.2', 'EntidadClave.3', 'ComunZonaRiesgoId', 'EIS_FolioId', 'EII_FolioId', 'ComunTipoConsultaId', 'CodigoPostal', 'Motivo_1', 'Motivo_2', 'Motivo_3', 'Motivo_4', 'Motivo_5', 'Motivo_6', 'Motivo_7', 'Motivo_8', 'Motivo_9','Motivo_10']
    list_col = [col for col in df.columns if col.startswith('SustanciaId')]
    list_col += [col for col in df.columns if col.startswith('ComunAbstinenciaId')]
    list_col += [col for col in df.columns if col.startswith('ComunPrevalenciaId')]
    list_col += [col for col in df.columns if col.startswith('OrdenConsumo')]
    list_col += [col for col in df.columns if col.startswith('Dosis')]
    list_col += [col for col in df.columns if col.startswith('ComunPrimeraFormaAdministracionId') or col.startswith('ComunSegundaFormaAdministracionId') or col.startswith('ComunTerceraFormaAdministracionId')]
    list_col += [col for col in df.columns if col.startswith('ComunUltimoConsumoId')]
    # list_col += [col for col in df.columns if col.startswith('EdadInicio')] ## Posible actualizacion para dejar estas columnas
    elimin += list_col
    df = df.drop(columns= elimin)
    return df

def OrderedColumns(df, denominador):
    '''
    Description: Esta funcion ordena las columnas del DataFrame de acuerdo con un orden predefinido basado en el denominador.
    Parameters:
    df (DataFrame): Un DataFrame de pandas que contiene los datos.
    denominador (str): El nombre del denominador que se utilizara para ordenar las columnas.
    Returns:
    DataFrame: Un DataFrame de pandas con las columnas ordenadas de acuerdo con el denominador.
    '''
    list_col = [col for col in df.columns if col.startswith('EdadInicio')]
    order = ['FolioId', 'SexoId', 'PerteneceComunidadLGBTTTI', 'PerteneceComunidadIndigena', 'PoblacionAfromexicanaAfroamericana', 'DiscapacidadPerceptual', 'ComunEstadoCivilId', 'ComunEscolaridadId', 'ComunOcupacionId', 'ComunEstratoSocialId', 'Migracion', 'Edad','Edad_Años', 'Año', 'Mes', 'Semestre', 'Estado', 'CentroCostoId' , 'Alcohol', 'Alucinógenos', 'Cocaína', 'Inhalables', 'Marihuana', 'Medicamentos', 'Metanfetaminas', 'Nuevas Sustancias Psicoactivas', 'Opioides', 'Otras Sustancias', 'Tabaco', 'ProblemasSalud', 'ProblemasFamiliares', 'AccidentesAsociados', 'ProblemasEscolares', 'ProblemasLaborales', 'ProblemasPsicologicos', 'ProblemasLegales', 'ConductaAntisocial','ConsumoDeDrogas', 'ConsumoDeBebidasAlcoholicas', 'ConsumoDeTabaco','Ludopatia', 'Otro', 'Depresion', 'Psicosis', 'Epilepsia','TrastornosMentales', 'Demencia', 'Autolesion', 'Suicidio', 'Ansiedad'] + list_col
    order2 = ['FolioId', 'SexoId', 'PerteneceComunidadLGBTTTI', 'PerteneceComunidadIndigena', 'PoblacionAfromexicanaAfroamericana', 'DiscapacidadPerceptual', 'ComunEstadoCivilId', 'ComunEscolaridadId', 'ComunOcupacionId', 'ComunEstratoSocialId', 'Migracion', 'Edad','Edad_Años', 'Año', 'Mes', 'Semestre', 'Estado','CentroCostoId', 'Tabaco', 'Nicotina en dispositivo electrónico', 'Bebidas Fermentadas', 'Bebidas Destiladas', 'Alcohol puro (alcohol del 96)', 'Mariguana estándar', 'Mariguana modificada, con alto nivel de THC',
            'Hachís - Resina', 'Hachís - Aceite o miel', 'Mariguana sintética', 'Medicamentos con compuestos cannábicos', 'Cocaína en polvo blanco', 'Crack','Basuco (pasta base de cocaína)', 'Otras presentaciones de cocaína', 'Metanfetaminas(cristal)', 'Metanfetaminas(hielo)', 'MetanfetaminasMetas', 'Sales de baño', 'Anfetaminas y derivados', 'Anorexígenos', 'Drogas de diseño con efectos estimulantes', 'Otros estimulantes', 'Solventes o removedores', 'Pegamentos', 'Esmaltes y pinturas', 'Gasolinas y combustibles',
            'Otros inhalantes', 'LSD', 'LSA', 'Hongos alucinógenos', 'Peyote (mescalina)', 'Salvia divinorum u otras variedades de salvia', 'Ololiuhqui', 'Floripondio', 'Ayahuasca', 'Otras plantas con efectos alucinógenos o disociativos', 'Fenciclidina (PCP, polvo de ángel)', 'Análogos del PCP', 'Ketamina', 'DMT', 'Triptaminas psicodélicas derivadas y análogas del DMT', 'Grupo 2Cx', 'Grupo DOx', 'Grupo NBOMe', 'Otras drogas de diseño con efectos psicodélicos', 'Otras sustancias con efectos psicodélicos', 'Éxtasis (MDMA)',
            'Drogas de diseño con efectos similares al éxtasis', 'Piperazinas con efectos similares al éxtasis', 'Aminoindanos', 'Benzodiacepinas y sustancias relacionadas', 'Rohypnol', 'GHB o similares', 'Sedantes hipnóticos', 'Otros depresores', 'Opio', 'Morfina(uso fuera de prescripción)', 'Codeína(uso fuera de prescripción)', 'Heroína blanca', 'Heroína negra', 'Oxicodona', 'Opiodes sintéticos (uso fuera de prescripción)', 'Fentanilo y análogos (uso fuera de prescripción)', 'Dextrometorfano (uso fuera de prescripción)', 'Derivados opiodes combinados con analgésicos',
            'Desomorfina', 'Anticolinérgicos', 'Antiespasmódicos', 'Antiparkinsonianos', 'Anticonvulsivos', 'Antidepresivos', 'Antipsicóticos', 'Antihistamínicos', 'Otras sustancias de abuso con utilidad médica', 'Nitritos', 'Anestésicos', 'Bebidas energizantes', 'Bebidas inteligentes', 'Esteroides anabólicos', 'Sustancias no clasificadas', 'Sustancias sin especificar', 'ProblemasSalud', 'ProblemasFamiliares', 'AccidentesAsociados', 'ProblemasEscolares', 'ProblemasLaborales', 'ProblemasPsicologicos', 'ProblemasLegales', 'ConductaAntisocial','ConsumoDeDrogas', 'ConsumoDeBebidasAlcoholicas', 'ConsumoDeTabaco','Ludopatia', 'Otro', 'Depresion', 'Psicosis', 'Epilepsia','TrastornosMentales', 'Demencia', 'Autolesion', 'Suicidio', 'Ansiedad' ] + list_col
    if 'Sust' in denominador:
        for col in order2:
            if col not in df.columns:
                df[col] = False
        dfnuevo = df[order2]
        return dfnuevo
    if 'les.csv' in denominador:
        dfnuevo = df[order]
        return dfnuevo

def Muestra_names(x):
    '''
    Description: Esta funcion convierte el nombre de una muestra en un formato abreviado.
    Parameters:
    x (str): El nombre de la muestra a convertir.
    Returns:
    str: El nombre de la muestra convertido en un formato abreviado.
    '''
    if x == 'DrogaImpacto':
        return 'DI'
    elif x == 'Algunavez':
        return 'AV'
    elif x == 'UltimoMes':
        return 'UM'

def Filt_name(x):
    '''
    Description: Esta funcion filtra el nombre de un archivo para determinar si es un archivo de sustancias o de drogas legales.
    Parameters:
    x (str): El nombre del archivo a filtrar.
    Returns:
    str: Un string que indica si el archivo es de sustancias o de drogas legales.
    '''
    if 'Sust' in x:
        return 'SUST'
    elif 'les.csv' in x:
        return 'GPOS'

def Denominador_names(x):
    '''
    Description: Esta funcion convierte el nombre de un denominador en un formato abreviado.
    Parameters:
    x (str): El nombre del denominador a convertir.
    Returns:
    str: El nombre del denominador convertido en un formato abreviado.
    '''
    if x == 'EntrevistaInicialDrogasIlegales.csv':
        return 'SI'
    elif x == 'EntrevistaInicialDrogasIlegalesSust.csv':
        return 'SI'
    elif x == 'EntrevistaInicialDrogasLegalesIlegales.csv':
        return 'SLI'
    elif x == 'EntrevistaInicialDrogasLegalesIlegalesSust.csv':
        return 'SLI'

def create_csv(process , muestra, denominador):
    '''
    Description: Esta funcion crea un archivo CSV a partir de un DataFrame y lo carga en un servidor FTP.
    Parameters:
    process (DataFrame): Un DataFrame de pandas que contiene los datos a guardar en el archivo CSV.
    muestra (str): El nombre de la muestra que se utilizara para nombrar el archivo CSV.
    denominador (str): El nombre del denominador que se utilizara para nombrar el archivo CSV.
    Returns:
    None
    '''
    muestra = Muestra_names(muestra)
    filt = Filt_name(denominador)
    denominador = Denominador_names(denominador)
    UploadCsvToFTP(process,f'/pruebas/csv/EntrevistaInicialSets/{filt}-{muestra}-{denominador}.csv')  

def Droga_Impacto(df, denominador):
    df = pd.get_dummies(df, columns= ['MayorImpactoSustanciaId'], prefix='', prefix_sep='')
    df = DropedColumns(df)
    df = OrderedColumns(df, denominador)
    return df

def Mod_data(df, muestra, denominador): 
    '''
    Description: Esta funcion procesa un DataFrame de pandas basado en la muestra y el denominador especificados, creando variables dummy, eliminando columnas innecesarias, ordenando las columnas y eliminando filas que no contienen datos sobre sustancias especificas.
    Parameters:
    df (DataFrame): Un DataFrame de pandas que contiene los datos a procesar.
    muestra (str): El nombre de la muestra que se utilizara para procesar el DataFrame.
    denominador (str): El nombre del denominador que se utilizara para procesar el DataFrame.
    Returns:
    None
    '''
    if muestra == 'DrogaImpacto':     
        df = Droga_Impacto(df, denominador)
        create_csv(df, muestra, denominador)

    elif muestra == 'Algunavez':
        lista_sustancias = [col for col in df.columns if col.startswith('SustanciaId')]
        value_to_column = extract_dem(denominador)
        for sust, nueva_col in value_to_column.items():
            if nueva_col not in df:
                df[nueva_col] = False
            df[nueva_col] = df[nueva_col] | df[lista_sustancias].isin([sust]).any(axis=1)
        df = DropedColumns(df)
        df = OrderedColumns(df, denominador)
        create_csv(df, muestra, denominador)

    elif muestra == 'UltimoMes':
        lista_sustancias = [col for col in df.columns if col.startswith('SustanciaId')]
        value_to_column = extract_dem(denominador)

        for nueva_col in value_to_column.values():
            if nueva_col not in df:
                df[nueva_col] = False

        for col in lista_sustancias:
            sustancia_id = re.search(r'SustanciaId(\d+)', col).group(1)
            filtered_group = df[
                (df[f'ComunUltimoConsumoId{sustancia_id}'].between(1, 3)) &
                (df[f'ComunAbstinenciaId{sustancia_id}'] == 1)
            ]

            for ind, val in filtered_group[col].items():
                if val in value_to_column.keys():
                    df.at[ind, value_to_column[val]] |= True  

        df = DropedColumns(df)
        df = OrderedColumns(df, denominador)
        create_csv(df, muestra, denominador)

In [44]:
def main():
    for name in ['EntrevistaInicialDrogasIlegales.csv' , 'EntrevistaInicialDrogasIlegalesSust.csv', 'EntrevistaInicialDrogasLegalesIlegales.csv', 'EntrevistaInicialDrogasLegalesIlegalesSust.csv']:
        df = ReadCsvFromFTP(f'/pruebas/csv/Recodificacion/EdadInicio/{name}')
        list_muestra = ['DrogaImpacto', 'Algunavez', 'UltimoMes']
        for muestra in list_muestra:
            print(f'Procesando {name} para {muestra}')
            df_copy = df.copy()
            Mod_data(df_copy, muestra, name)

In [45]:
main()

Conectado a 192.168.62.13


C:\Users\franc\AppData\Local\Temp\ipykernel_14328\1755963303.py:49: DtypeWarning: Columns (31,50,51,52,53,54,70,71,72,73,74,75,76,77,78,79,80,81,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,296,304,318,319,320,321,322,323,338,339,342,343,344,345,346,347,390,391) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(bio)


Procesando EntrevistaInicialDrogasIlegales.csv para DrogaImpacto
Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/EntrevistaInicialSets/GPOS-DI-SI.csv
Procesando EntrevistaInicialDrogasIlegales.csv para Algunavez
Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/EntrevistaInicialSets/GPOS-AV-SI.csv
Procesando EntrevistaInicialDrogasIlegales.csv para UltimoMes
Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/EntrevistaInicialSets/GPOS-UM-SI.csv
Conectado a 192.168.62.13


C:\Users\franc\AppData\Local\Temp\ipykernel_14328\1755963303.py:49: DtypeWarning: Columns (31,50,51,52,53,54,70,71,72,73,74,75,76,77,78,79,80,81,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,296,304,318,319,320,321,322,323,338,339,342,343,344,345,346,347,390,391) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(bio)


Procesando EntrevistaInicialDrogasIlegalesSust.csv para DrogaImpacto
Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/EntrevistaInicialSets/SUST-DI-SI.csv
Procesando EntrevistaInicialDrogasIlegalesSust.csv para Algunavez
Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/EntrevistaInicialSets/SUST-AV-SI.csv
Procesando EntrevistaInicialDrogasIlegalesSust.csv para UltimoMes
Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/EntrevistaInicialSets/SUST-UM-SI.csv
Conectado a 192.168.62.13


C:\Users\franc\AppData\Local\Temp\ipykernel_14328\1755963303.py:49: DtypeWarning: Columns (31,50,51,52,53,54,70,71,72,73,74,75,76,77,78,79,80,81,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,296,304,318,319,320,321,322,323,338,339,342,343,344,345,346,347,390,391) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(bio)


Procesando EntrevistaInicialDrogasLegalesIlegales.csv para DrogaImpacto
Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/EntrevistaInicialSets/GPOS-DI-SLI.csv
Procesando EntrevistaInicialDrogasLegalesIlegales.csv para Algunavez
Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/EntrevistaInicialSets/GPOS-AV-SLI.csv
Procesando EntrevistaInicialDrogasLegalesIlegales.csv para UltimoMes
Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/EntrevistaInicialSets/GPOS-UM-SLI.csv
Conectado a 192.168.62.13


C:\Users\franc\AppData\Local\Temp\ipykernel_14328\1755963303.py:49: DtypeWarning: Columns (31,50,51,52,53,54,70,71,72,73,74,75,76,77,78,79,80,81,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,296,304,318,319,320,321,322,323,338,339,342,343,344,345,346,347,390,391) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(bio)


Procesando EntrevistaInicialDrogasLegalesIlegalesSust.csv para DrogaImpacto
Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/EntrevistaInicialSets/SUST-DI-SLI.csv
Procesando EntrevistaInicialDrogasLegalesIlegalesSust.csv para Algunavez
Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/EntrevistaInicialSets/SUST-AV-SLI.csv
Procesando EntrevistaInicialDrogasLegalesIlegalesSust.csv para UltimoMes
Conectado a 192.168.62.13
Archivo subido correctamente: /pruebas/csv/EntrevistaInicialSets/SUST-UM-SLI.csv
